In [ ]:
# Standard-library and package imports used throughout the notebook.
import os
import certifi
import requests

from dotenv import load_dotenv

# LangChain integrations for the chat model, tools, and prompt hub.
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool



# Notebook overview
This notebook builds a LangChain ReAct agent that can answer questions using an OpenAI chat model and Tavily web search. It loads API credentials, checks the search tool independently, fetches a ReAct prompt from LangSmith, connects the model and tool into an agent, then runs a question that requires current information.

The main flow is: configure credentials -> test search -> set up the model and prompt -> create the agent -> run it through `AgentExecutor`.


In [14]:
from langchain.agents import create_react_agent, AgentExecutor

## What does `create_react_agent` do?
`create_react_agent` builds an agent that follows the ReAct pattern: the language model reasons about the request, chooses whether to call one of the available tools, receives the tool result, and then continues until it can produce a final response. It combines a model, tools, and a prompt into an agent runnable.

In this notebook, this import comes from `langchain.agents`. In older LangChain versions, the runnable is commonly passed to `AgentExecutor` to manage the tool-call loop. This is separate from LangGraph's `create_react_agent`, which is imported from `langgraph.prebuilt`.

## Pulling a prompt from LangSmith
The older `langchain.hub` helper emits a deprecation warning. Use the LangSmith client instead. Also use the current ReAct prompt identifier, `hwchase17/react`; the old `hwchase/react` identifier can return a 404.

```python
from langsmith import Client

prompt = Client().pull_prompt("hwchase17/react")
```

If this prompt is unavailable to your account, verify the prompt name/access in LangSmith or define a compatible ReAct prompt locally.


In [50]:
# =======================
# LOAD ENV VARIABLES
# ======================

# Use certifi's trusted certificate bundle for HTTPS requests.
os.environ["SSL_CERT_FILE"] = certifi.where()

# Read API keys from the .env file into the process environment.
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY=os.getenv("WEATHERSTACK_API_KEY")

### Load credentials and configure HTTPS
This setup loads the keys from the local `.env` file so the OpenAI and Tavily clients can authenticate. Setting `SSL_CERT_FILE` to certifi's bundle helps Python verify HTTPS certificates when connecting to those services.


In [ ]:
# Configure Tavily to return at most two web-search results per query.
#  this is a toolalready created as a library / package 
search_tool = TavilySearchResults(max_results=2)


In [ ]:
# ============================================

# my custom function that i will use as a tool 
#  CUSTOM TOOL 
# ==============================================

@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"http://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )



In [38]:
# Test the search tool directly before giving it to the agent.
result = search_tool.invoke("What is the latest news on Ai")


In [26]:
print(result)

[{'url': 'https://guides.library.georgetown.edu/ai/news', 'content': 'Skip to Main Content\n\n# Artificial Intelligence (Generative) Resources: Latest News on AI\n\n Home\n AI Fundamentals\n AI Tools for Research\n How to Craft Prompts\n Research Resources on AI\n Latest News on AI \n  + AI News from the New York Times, WSJ, and Washington Post (ProQuest)\n  + AI News from Education Periodicals (ProQuest)\n Ethics & AI\n Citing Generative AI\n AI, Authorship, & Copyright\n Campus Resources and Policies\n\n## Publications covering AI\n\n Technology Review This link opens in a new window  \n\n  Formerly MIT Technology Review. Includes The Algorithm.\n VKTR: The Business of Enterprise AI This link opens in a new window'}, {'url': 'https://www.sciencedaily.com/news/computers_math/artificial_intelligence', 'content': '### Monday, May 18, 2026\n\n Forget Electrons, This Breakthrough Uses Light-Matter Particles to Power AI\n\n### Friday, May 15, 2026\n\n NASA’s New AI Space Chip Could Let Spa

### Test web search and initialize the model
The previous cells call Tavily directly and display its results, confirming that the search tool works on its own before it is attached to an agent. The following model setup creates the LLM that will decide whether to use the tool and compose the final response.


In [45]:
# Configure the chat model used by the agent.
llm = ChatOpenAI(
    model = "gpt-4.1-mini" ,
    temperature=0,
    api_key=OPENAI_API_KEY
)

# Make a simple test call to confirm the model connection works.
llm.invoke("What year is it")

# A model's knowledge can be out of date; use web search for current facts.


AIMessage(content='The current year is 2024. How can I assist you today?', response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 11, 'total_tokens': 26, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini', 'system_fingerprint': 'fp_41f1fe1393', 'finish_reason': 'stop', 'logprobs': None}, id='run-f95e725d-4e01-45a5-9fc4-25f025024d2f-0')

In [28]:
# Instantiate the same model configuration as a standalone object.
ChatOpenAI(model="gpt-4.1-mini")


ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001D89BC07C50>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001D89BB95590>, model_name='gpt-4.1-mini', openai_api_key=SecretStr('**********'), openai_proxy='')

In [29]:
# Download the ReAct prompt template from LangSmith for the agent to use.
from langsmith import Client

prompt = Client().pull_prompt("hwchase17/react")


c:\Users\rasto\.conda\envs\langagent\Lib\site-packages\langsmith\client.py:241: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


### What this prompt-loading code does
`Client()` creates a LangSmith client, and `pull_prompt("hwchase17/react")` downloads the named ReAct prompt template. It does not send a question to the model or generate a response. The returned prompt can be supplied to `create_react_agent` along with the LLM and tools. Pulling it requires internet access and permission to access that prompt.


### How the `hwchase17/react` prompt guides the agent
This prompt defines a ReAct loop: the model receives the question and available tools, then chooses an action. `Action` names a tool, `Action Input` supplies that tool's input, and `Observation` is the tool's result. The agent can repeat this cycle until it has enough information to return a `Final Answer`.

The placeholders are filled by the agent setup: `{tools}` lists available tools, `{tool_names}` lists valid tool names, `{input}` contains the user's question, and `{agent_scratchpad}` carries the intermediate action/observation history so the model can continue the loop. The prompt is an instruction template; `create_react_agent` and its executor interpret the model's tool actions and run the tools.

Here is the prompt template for reference:

```text
Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}
```


### Put the components together
The next cells make the search tool available to the agent, create a ReAct agent from the LLM, tools, and prompt, and wrap it in `AgentExecutor`. The final invocation asks for India's capital and Meerut's current weather, so the agent can demonstrate searching for fresh facts before answering.


In [46]:
# Give the agent the Tavily search tool it is allowed to call.

tools = [
    search_tool,
    get_weather_data
]


In [47]:
# Combine the model, available tools, and prompt into a ReAct agent.
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)


In [48]:
# Wrap the agent with the executor that runs its tool-call loop.
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)


### What is an `AgentExecutor`?
`AgentExecutor` runs the agent together with its tools. It sends the request to the agent, executes any tool actions the agent chooses, returns each tool result to the agent, and repeats until the agent produces a final answer or reaches its execution limit. The executor is the runtime loop around the agent; the agent itself decides what action to take.

Here, `tools=tools` provides the tools the executor is allowed to run, and `verbose=True` prints intermediate steps to help you see the process while developing.


In [51]:
# =====================================
# RUN
# ==============================

# Run the agent on a question that needs a factual answer and current weather.
response = agent_executor.invoke({
    "input": (
        "Find the capital of India"
        "and then find Meerut's current weather."
    )
})

# Display the final answer returned by the executor.
print("\n========================")
print("FINAL OUTPUT")
print("========================\n")

print(response["output"])




> Entering new AgentExecutor chain...
Thought: First, I need to find the capital of India. Then, I will get the current weather for Meerut.
Action: tavily_search_results_json
Action Input: capital of India[{'url': 'https://www.vedantu.com/question-answer/what-is-the-capital-of-india-692ffa0b993a9249b7ff7708', 'content': "banner\n\nWhat is the capital of India?\n\nAnswer\n\nVerifiedVerified\n\n94.8k+views\n\nAnswer: New Delhi\n\n  \n\nExplanation:\n\nNew Delhi is the capital city of India and serves as the seat of the Government of India. Located in the northern part of the country, New Delhi is part of the larger National Capital Territory of Delhi, which also includes Old Delhi and several other areas.\n\n  \n\nThe city was officially designated as India's capital in 1911 during British rule, replacing Calcutta (now Kolkata). The British decided to shift the capital from Calcutta to Delhi due to its more central location and historical significance. New Delhi was specifically planne